# ADSP 31014 — Statistical Models for Data Science
## Homework 5

**Name:** Obaro Erigha
**Term:** Autumn 2026

In [1]:
data <- read.csv('../../Week3/Hw3/chicago_taxi_trip_sample.csv')
dim(data)

# hour_type as defined in the lecture code
hour_type <- list('Rush Hour' = c(7:9, 15:18),
                  'Night'     = c(0:6, 21:23),
                  'Other'     = c(10:14, 19:20))
data$hour_type <- ''
for (i in seq_along(hour_type)) {
  data$hour_type[data$hour %in% hour_type[[i]]] <- names(hour_type)[i]
}
table(data$hour_type)

# Prints the model size, AIC and adjusted R^2 on one line
fit.stats <- function(model) {
  cat(sprintf('betas = %d, AIC = %.3f, adjusted R^2 = %.7f
', length(coef(model)),
              AIC(model), summary(model)$adj.r.squared))
}

[1] 100000     17


    Night     Other Rush Hour 
    17184     42043     40773 

#### Question 1a. Initial Model:

In [2]:
model.1a <- lm(trip_seconds ~ extras + fare + tips + tips_percent + tolls
               + trip_miles + trip_total, data = data)
summary(model.1a)
fit.stats(model.1a)


Call:
lm(formula = trip_seconds ~ extras + fare + tips + tips_percent + 
    tolls + trip_miles + trip_total, data = data)

Residuals:
    Min      1Q  Median      3Q     Max 
-8061.5  -208.1   -49.5   146.5  6326.3 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept)   159.2488     3.5120  45.345  < 2e-16 ***
extras         16.2973     5.0556   3.224 0.001266 ** 
fare           68.1189     4.8933  13.921  < 2e-16 ***
tips           17.2839     5.0473   3.424 0.000616 ***
tips_percent -132.6507    13.4249  -9.881  < 2e-16 ***
tolls           7.1237    22.0903   0.322 0.747088    
trip_miles     -9.3136     0.5992 -15.544  < 2e-16 ***
trip_total    -15.1102     4.9127  -3.076 0.002100 ** 
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 511.5 on 99992 degrees of freedom
Multiple R-squared:  0.6807,	Adjusted R-squared:  0.6806 
F-statistic: 3.045e+04 on 7 and 99992 DF,  p-value: < 2.2e-16


betas = 8, AIC = 1531258.150, adjusted R^2 = 0.6806288


AIC = 1531258.150

Adjusted R^2 = 0.6806288

tolls is the least useful explanatory variable based on the t-test, with p-value = 0.7471.

#### Question 1b. Backward Elimination:

In [3]:
model.1b <- step(model.1a, direction = 'backward')
summary(model.1b)
fit.stats(model.1b)

Start:  AIC=1247468
trip_seconds ~ extras + fare + tips + tips_percent + tolls + 
    trip_miles + trip_total

               Df Sum of Sq        RSS     AIC
- tolls         1     27206 2.6159e+10 1247467
<none>                      2.6159e+10 1247468
- trip_total    1   2474836 2.6161e+10 1247476
- extras        1   2718562 2.6161e+10 1247477
- tips          1   3067776 2.6162e+10 1247478
- tips_percent  1  25541764 2.6184e+10 1247564
- fare          1  50696334 2.6209e+10 1247660
- trip_miles    1  63208222 2.6222e+10 1247708

Step:  AIC=1247467
trip_seconds ~ extras + fare + tips + tips_percent + trip_miles + 
    trip_total

               Df Sum of Sq        RSS     AIC
<none>                      2.6159e+10 1247467
- trip_total    1   2486248 2.6161e+10 1247474
- extras        1   2736796 2.6162e+10 1247475
- tips          1   3093143 2.6162e+10 1247476
- tips_percent  1  25638123 2.6184e+10 1247563
- fare          1  52861115 2.6212e+10 1247666
- trip_miles    1  63298059 2.6222


Call:
lm(formula = trip_seconds ~ extras + fare + tips + tips_percent + 
    trip_miles + trip_total, data = data)

Residuals:
    Min      1Q  Median      3Q     Max 
-8055.1  -208.1   -49.5   146.5  6326.3 

Coefficients:
             Estimate Std. Error t value Pr(>|t|)    
(Intercept)   159.210      3.510  45.360  < 2e-16 ***
extras         15.929      4.925   3.234 0.001219 ** 
fare           67.763      4.767  14.215  < 2e-16 ***
tips           16.923      4.921   3.439 0.000585 ***
tips_percent -132.810     13.416  -9.900  < 2e-16 ***
trip_miles     -9.318      0.599 -15.555  < 2e-16 ***
trip_total    -14.751      4.785  -3.083 0.002051 ** 
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 511.5 on 99993 degrees of freedom
Multiple R-squared:  0.6807,	Adjusted R-squared:  0.6806 
F-statistic: 3.552e+04 on 6 and 99993 DF,  p-value: < 2.2e-16


betas = 7, AIC = 1531256.254, adjusted R^2 = 0.6806317


Note: step() prints AIC from extractAIC(), which drops a constant, so its numbers are on a different scale from AIC(). The ranking of models is the same. The values reported here come from AIC().

Reduced model: trip_seconds ~ extras + fare + tips + tips_percent + trip_miles + trip_total

Only tolls was eliminated.

AIC = 1531256.254

Adjusted R^2 = 0.6806317

trip_total is the least useful remaining variable based on the t-test, with p-value = 0.002051.

We should not eliminate it based on this p-value. 0.002051 is well below 0.05, so we reject H0 that its coefficient is 0. AIC agrees, since removing it would raise AIC.

#### Question 1c. Forward Selection:

In [4]:
null.model <- lm(trip_seconds ~ 1, data = data)
model.1c <- step(null.model, scope = formula(model.1a), direction = 'forward')
summary(model.1c)
fit.stats(model.1c)

Start:  AIC=1361602
trip_seconds ~ 1

               Df  Sum of Sq        RSS     AIC
+ fare          1 5.5639e+10 2.6273e+10 1247894
+ trip_total    1 5.2629e+10 2.9284e+10 1258742
+ trip_miles    1 4.3876e+10 3.8037e+10 1284893
+ extras        1 1.4910e+10 6.7003e+10 1341512
+ tips          1 1.4688e+10 6.7225e+10 1341843
+ tips_percent  1 4.8368e+08 8.1429e+10 1361011
+ tolls         1 3.2284e+07 8.1881e+10 1361564
<none>                       8.1913e+10 1361602

Step:  AIC=1247894
trip_seconds ~ fare

               Df Sum of Sq        RSS     AIC
+ trip_miles    1  69225488 2.6204e+10 1247632
+ tips_percent  1  44601605 2.6229e+10 1247726
+ tips          1  18974725 2.6254e+10 1247824
+ trip_total    1  17347408 2.6256e+10 1247830
+ extras        1   1119090 2.6272e+10 1247891
<none>                      2.6273e+10 1247894
+ tolls         1     60930 2.6273e+10 1247896

Step:  AIC=1247632
trip_seconds ~ fare + trip_miles

               Df Sum of Sq        RSS     AIC
+ tips_perce


Call:
lm(formula = trip_seconds ~ fare + trip_miles + tips_percent + 
    tips, data = data)

Residuals:
    Min      1Q  Median      3Q     Max 
-8082.9  -207.6   -49.7   146.3  6327.2 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept)   157.8372     3.4837  45.308  < 2e-16 ***
fare           53.1163     0.2671 198.898  < 2e-16 ***
trip_miles     -9.4749     0.5935 -15.965  < 2e-16 ***
tips_percent -139.7508    13.2435 -10.552  < 2e-16 ***
tips            2.1674     0.7810   2.775  0.00552 ** 
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 511.5 on 99995 degrees of freedom
Multiple R-squared:  0.6806,	Adjusted R-squared:  0.6806 
F-statistic: 5.327e+04 on 4 and 99995 DF,  p-value: < 2.2e-16


betas = 5, AIC = 1531262.845, adjusted R^2 = 0.6806042


Selected model: trip_seconds ~ fare + trip_miles + tips_percent + tips

AIC = 1531262.845

Adjusted R^2 = 0.6806042

This is not the same as the reduced model in 1b. Forward selection stops with 4 explanatory variables, while backward elimination keeps those 4 plus extras and trip_total.

#### Question 1d. Borderline Variables:

In [5]:
model.1d <- update(model.1b, . ~ . - trip_total)
summary(model.1d)
fit.stats(model.1d)


Call:
lm(formula = trip_seconds ~ extras + fare + tips + tips_percent + 
    trip_miles, data = data)

Residuals:
    Min      1Q  Median      3Q     Max 
-8080.1  -207.6   -49.7   146.2  6327.3 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept)   157.8981     3.4842  45.319   <2e-16 ***
extras          1.0974     1.0525   1.043   0.2971    
fare           53.0901     0.2682 197.923   <2e-16 ***
tips            1.9560     0.8069   2.424   0.0153 *  
tips_percent -138.8021    13.2747 -10.456   <2e-16 ***
trip_miles     -9.5238     0.5953 -15.998   <2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 511.5 on 99994 degrees of freedom
Multiple R-squared:  0.6806,	Adjusted R-squared:  0.6806 
F-statistic: 4.262e+04 on 5 and 99994 DF,  p-value: < 2.2e-16


betas = 6, AIC = 1531263.758, adjusted R^2 = 0.6806045


In [6]:
# extras coefficient and p-value before and after removing trip_total
rbind(with.trip_total    = summary(model.1b)$coefficients['extras', ],
      without.trip_total = summary(model.1d)$coefficients['extras', ])

# Dropping extras as well gives the forward selection model from 1c
model.1d.drop <- update(model.1d, . ~ . - extras)
all.equal(fitted(model.1d.drop), fitted(model.1c))

,Estimate,Std. Error,t value,Pr(>|t|)
with.trip_total,15.928882,4.924794,3.234426,0.001219271
without.trip_total,1.097396,1.052546,1.042611,0.297131095


[1] TRUE

Further reduced model: trip_seconds ~ extras + fare + tips + tips_percent + trip_miles

AIC = 1531263.758

Adjusted R^2 = 0.6806045

extras suddenly becomes not very useful. Its p-value based on the t-test goes from 0.001219 to 0.2971.

trip_total is close to fare + tips + tolls + extras, so these variables overlap heavily. extras and trip_total were only useful together, which is why backward elimination keeps both and forward selection never adds either one. Removing extras here lands on exactly the forward selection model from 1c.

#### Question 1e. Best Model:

In [7]:
models <- list('1a full' = model.1a, '1b reduced' = model.1b,
               '1c forward' = model.1c, '1d further reduced' = model.1d)
comparison <- data.frame(betas  = sapply(models, function(m) length(coef(m))),
                         AIC    = sprintf('%.3f', sapply(models, AIC)),
                         adj.R2 = sprintf('%.7f', sapply(models, function(m) summary(m)$adj.r.squared)))
comparison

,betas,AIC,adj.R2
,<int>,<chr>,<chr>
1a full,8,1531258.150,0.6806288
1b reduced,7,1531256.254,0.6806317
1c forward,5,1531262.845,0.6806042
1d further reduced,6,1531263.758,0.6806045


The reduced model from 1b is the best on both criteria: lowest AIC (1531256.254) and highest adjusted R^2 (0.6806317).

#### Question 2a. "I'm Lazy" Model:

In [8]:
model.2a <- lm(trip_seconds ~ fare * factor(day_of_week) * hour_type
               * pickup_area * dropoff_area * payment_type, data = data)

length(coef(model.2a))      # beta parameters in the formula
sum(is.na(coef(model.2a)))  # not estimable (NA)
model.2a$rank               # beta parameters actually estimated
fit.stats(model.2a)

[1] 2016

[1] 253

[1] 1763

betas = 2016, AIC = 1497798.473, adjusted R^2 = 0.7753902


The formula has 2016 beta parameters: 2 x 7 x 3 x 4 x 4 x 3 = 2016 (fare gives an intercept and a slope, then the levels of day_of_week, hour_type, pickup_area, dropoff_area and payment_type multiply).

253 of these come back as NA, so only 1763 beta parameters are actually estimated. The model fits a separate line (intercept and fare slope) for each of the 7 x 3 x 4 x 4 x 3 = 1008 combinations of the categorical variables. 104 combinations have no trips, which loses 2 parameters each, and 45 have only one distinct fare, so no slope can be estimated, which loses 1 each: 104 x 2 + 45 = 253.

AIC = 1497798.473

Adjusted R^2 = 0.7753902

#### Question 2b. Hope or Hopeless:

In [9]:
options(width = 200)  # the term name is very long
model.2b <- step(model.2a, direction = 'backward', test = 'F')

length(coef(model.2b))
model.2b$rank
all.equal(formula(model.2b), formula(model.2a))

Start:  AIC=1214009
trip_seconds ~ fare * factor(day_of_week) * hour_type * pickup_area * 
    dropoff_area * payment_type

                                                                            Df Sum of Sq        RSS     AIC F value    Pr(>F)    
<none>                                                                                   1.8074e+10 1214009                      
- fare:factor(day_of_week):hour_type:pickup_area:dropoff_area:payment_type 128 103457871 1.8178e+10 1214324  4.3931 < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1


[1] 2016

[1] 1763

[1] TRUE

No variables or terms are eliminated. The model still has 2016 beta parameters (1763 estimated), the same as 2a, so in this case backward elimination is hopeless.

step() respects the hierarchy of the terms, so the only term it can consider dropping is the highest order 6-way interaction. That term has F = 4.3931 on 128 degrees of freedom with p-value < 2.2e-16, so we reject H0 and keep it. AIC also goes up if it is removed. Once the 6-way interaction stays, none of the lower order terms can be removed, so elimination stops at the first step.

With 100,000 data points, the F-test finds even this complicated interaction statistically significant. Question 3 shows why.

#### Question 3a. Large Data Set vs Tiny Effect:

In [10]:
set.seed(1)

num.of.repeat <- 10
p.values <- numeric(num.of.repeat)
for (i in seq_len(num.of.repeat)) {
  # Step 1: Simulate observed data x_1,...,x_n ~ Bern(p1) under H1
  n <- 1e5
  p1 <- 0.5 + 1e-5
  x <- rbinom(n = 1, size = n, prob = p1)
  # Step 2: Compute p-value of two-sided binomial proportion test H0: p = 0.5
  p.values[i] <- prop.test(x, n, p = 0.5)$p.value
}

max(p.values)

[1] 0.9621672

prop.test(x, n, p = 0.5) completes the code.

max(p.values) = 0.9621672 (with set.seed(1))

At n = 10^5 the p-value is nowhere near small, so we cannot detect the unfair coin.

#### Question 3b. Small P-Value:

In [11]:
max.p.value <- function(n, num.of.repeat = 10) {
  p.values <- numeric(num.of.repeat)
  for (i in seq_len(num.of.repeat)) {
    p1 <- 0.5 + 1e-5
    x <- rbinom(n = 1, size = n, prob = p1)
    p.values[i] <- prop.test(x, n, p = 0.5)$p.value
  }
  max(p.values)
}

set.seed(1)
k <- 5:12
data.frame(k = k, n = 10^k, max.p.value = sapply(10^k, max.p.value))

k,n,max.p.value
<int>,<dbl>,<dbl>
5,1e+05,9.621672e-01
6,1e+06,9.497665e-01
7,1e+07,9.397545e-01
8,1e+08,9.723190e-01
9,1e+09,8.338104e-01
10,1e+10,6.879711e-01
11,1e+11,3.296674e-05
12,1e+12,1.161708e-79


n must be about 10^11 (k = 11), where max(p.values) = 3.296674e-05 < 0.01. At n = 10^10 it is still 0.6879711.

This makes sense: the test statistic is about 2 x 10^-5 x sqrt(n) on average. That is only 2 at n = 10^10 but about 6.3 at n = 10^11, which is large enough for all 10 repeats to be significant.